# Projet NLP : Named entity recognition du titulaire / suppléant et de son affiliation politique

# Objectifs

On souhaite évaluer la performance d'une méthode Named-Entity-Recognition pour l'identification du titulaire de la profession de foi et de son affiliation politique. Pour cela, on tire parti des annotations disponibles, pour tagger les noms des titulaires / suppléants dans les transcriptions.

Plus précisément, on évalue ici la performance de GlinNER2 car : 
- c'est un modèle de type transformer de petite taille, uniquement CPU
- le modèle permet de faire du zero-shot learning et permet d'indiquer que l'on souhaite avoir l'affiliation politique correspondante à l'entité reconnue

# Annotations / création d'un jeu de test

On annote les professions de foi à partir des métadonnées disponibles.
Résulat : 
- liste des segments de texte où apparaissent le nom du titulaire / suppléant 
- affiliation politique de la profession de foi

## Lecture des métadonnées

In [1]:
import contextlib
import pandas as pd
import polars as pl
from tqdm import tqdm

In [2]:
# read with pandas, more permissive on CSV parsing
csv = pd.read_csv("archelect_search.csv")
csv['date'] = pd.to_datetime(csv['date'])

/tmp/ipykernel_10787/1506034714.py:2: DtypeWarning: Columns (0: departement-nom, 1: departement-insee, 2: identifiant de circonscription, 3: pdf, 4: suppleant-nom, 5: suppleant-prenom, 6: suppleant-sexe, 7: suppleant-age, 8: suppleant-age-calcule, 9: suppleant-age-tranche, 10: suppleant-profession, 11: suppleant-mandat-en-cours, 12: suppleant-mandat-passe, 13: suppleant-associations, 14: suppleant-autres-statuts, 15: suppleant-soutien, 16: suppleant-liste, 17: suppleant-decorations) have mixed types. Specify dtype option on import or set low_memory=False.
  csv = pd.read_csv("archelect_search.csv")


In [3]:
# select 1988 election (because we only have transcriptions for this election)
csv1988 = csv.loc[(csv['date'].dt.year==1988) & (csv['contexte-election']=='législatives')]

In [4]:
csv1988 = pl.from_pandas(csv1988)

In [5]:
# select only rows with non trivial titulaire
csv1988.get_column("titulaire-nom").value_counts().sort("count") # 79 valeurs "non mentionné"
csv1988.with_columns(
    pl.col("titulaire-nom").str.len_chars().alias("titulaire-nom-length")
).sort("titulaire-nom-length").select(pl.selectors.by_name("titulaire-nom","titulaire-nom-length")) # pas de chaîne vide
csv1988 = csv1988.filter(pl.col("titulaire-nom") != "non mentionné") # 3461 rows

## Lecture des transcripts

In [6]:
def read_transcript(id):
    try:
        with open("text_files/1988/legislatives/"+id+".txt", "r") as file:
            return file.read()
    except FileNotFoundError:
        return '' # transcript not found, return empty string

csv1988 = csv1988.with_columns(
    pl.col("id").map_elements(read_transcript, return_dtype=pl.String).alias("transcript")
)
csv1988.filter(pl.col("transcript") == '') # only 1 missing transcript
csv1988 = csv1988.filter(pl.col("transcript") != '') # 3460 rows

In [7]:
csv1988

id,date,subject,title,contexte-election,contexte-tour,cote,departement,departement-nom,departement-insee,identifiant de circonscription,images,pdf,ocr_url,titulaire-nom,titulaire-prenom,titulaire-sexe,titulaire-age,titulaire-age-calcule,titulaire-age-tranche,titulaire-profession,titulaire-mandat-en-cours,titulaire-mandat-passe,titulaire-associations,titulaire-autres-statuts,titulaire-soutien,titulaire-liste,titulaire-decorations,suppleant-nom,suppleant-prenom,suppleant-sexe,suppleant-age,suppleant-age-calcule,suppleant-age-tranche,suppleant-profession,suppleant-mandat-en-cours,suppleant-mandat-passe,suppleant-associations,suppleant-autres-statuts,suppleant-soutien,suppleant-liste,suppleant-decorations,transcript
str,datetime[μs],str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""EL174_L_1988_06_001_01_1_PF_01""",1988-06-05 00:00:00,"""France;Assemblée Nationale;Ve …","""Élections législatives de 1988…","""législatives""",1,"""EL174""","""01""","""Ain""","""01 - Ain""","""1""","""https://ia803409.us.archive.or…","""https://ia803409.us.archive.or…","""https://ia803409.us.archive.or…","""Aulagne""","""Bernard""","""homme""","""45""","""45""","""entre 40 et 49 ans""","""marbrier""","""non mentionné""","""non mentionné""","""local""","""non mentionné""","""Front national""","""Liste d'entente populaire et n…","""non""","""Le Faucheur""","""Tiphaine""","""femme""","""non mentionné""","""non mentionné""","""non mentionné""","""non mentionné""","""non mentionné""","""non mentionné""","""non mentionné""","""non mentionné""","""Front national""","""Liste d'entente populaire et n…","""non""","""Élections Législatives du 5 Ju…"
"""EL174_L_1988_06_001_01_1_PF_02""",1988-06-05 00:00:00,"""Élections législatives;Assembl…","""Élections législatives de 1988…","""législatives""",1,"""EL174""","""01""","""Ain""","""01 - Ain""","""1""","""https://ia803400.us.archive.or…","""https://ia803400.us.archive.or…","""https://ia803400.us.archive.or…","""Pujol""","""Régis""","""homme""","""58""","""58""","""entre 50 et 59 ans""","""non mentionné""","""non mentionné""","""non mentionné""","""non mentionné""","""chômeur""","""Comités Juquin""","""Une nouvelle politique à gauch…","""non""","""Mortel""","""Jean-François""","""homme""","""non mentionné""","""non mentionné""","""non mentionné""","""non mentionné""","""non mentionné""","""non mentionné""","""non mentionné""","""non mentionné""","""Comités Juquin""","""Une nouvelle politique à gauch…","""non""","""RÉPUBLIQUE FRANÇAISE - Liberté…"
"""EL174_L_1988_06_001_01_1_PF_03""",1988-06-05 00:00:00,"""Assemblée Nationale;Ve Républi…","""Élections législatives de 1988…","""législatives""",1,"""EL174""","""01""","""Ain""","""01 - Ain""","""1""","""https://ia801800.us.archive.or…","""https://ia801800.us.archive.or…","""https://ia801800.us.archive.or…","""Boyon""","""Jacques""","""homme""","""non mentionné""","""non mentionné""","""non mentionné""","""non mentionné""","""conseiller général;maire""","""secrétaire d'Etat""","""non mentionné""","""non mentionné""","""Rassemblement pour la Républiq…","""Union du rassemblement et du c…","""oui""","""Morin""","""Paul""","""homme""","""non mentionné""","""non mentionné""","""non mentionné""","""non mentionné""","""conseiller général""","""maire-adjoint honoraire""","""non mentionné""","""non mentionné""","""Rassemblement pour la Républiq…","""Union du rassemblement et du c…","""oui""","""Sciences Po / fonds CEVIPOF EL…"
"""EL174_L_1988_06_001_01_1_PF_04""",1988-06-05 00:00:00,"""Assemblée Nationale;Ve Républi…","""Élections législatives de 1988…","""législatives""",1,"""EL174""","""01""","""Ain""","""01 - Ain""","""1""","""https://ia801807.us.archive.or…","""https://ia801807.us.archive.or…","""https://ia801807.us.archive.or…","""Saint-Pierre""","""Dominique""","""homme""","""47""","""47""","""entre 40 et 49 ans""","""avocat""","""conseiller régional""","""d

## Normalisation du nom

In [8]:
df = csv1988

In [9]:
df = df.with_columns(
    pl.col("transcript").str.to_lowercase()
      .str.normalize("NFKD")
      .str.replace_all(r"\p{CombiningMark}", "").alias("transcript-normalized"),
    pl.col("titulaire-nom").str.to_lowercase()
      .str.normalize("NFKD")
      .str.replace_all(r"\p{CombiningMark}", "").alias("titulaire-nom-normalized"),
)

In [10]:
df.select(pl.selectors.by_name("titulaire-nom","titulaire-nom-normalized"))

titulaire-nom,titulaire-nom-normalized
str,str
"""Aulagne""","""aulagne"""
"""Pujol""","""pujol"""
"""Boyon""","""boyon"""
"""Saint-Pierre""","""saint-pierre"""
"""Mornet""","""mornet"""
…,…
"""Hoarau""","""hoarau"""
"""Pihouee""","""pihouee"""
"""Hoarau""","""hoarau"""


## Positions du nom dans le transcript

In [11]:
import re

def find_aux(row):
    return [match.start() for match in re.finditer(row["pattern"], row["text"])]

df = df.with_columns(
     pl.struct(["transcript-normalized","titulaire-nom-normalized"])
        .struct.rename_fields(["text","pattern"])
        .map_elements(find_aux,return_dtype=pl.List(pl.Int64))
        .alias("titulaire-nom-positions")
)

In [12]:
df.select(pl.selectors.by_name("transcript","titulaire-nom-normalized","titulaire-nom-positions"))

transcript,titulaire-nom-normalized,titulaire-nom-positions
str,str,list[i64]
"""Élections Législatives du 5 Ju…","""aulagne""","[1240, 2866]"
"""RÉPUBLIQUE FRANÇAISE - Liberté…","""pujol""","[208, 3567]"
"""Sciences Po / fonds CEVIPOF EL…","""boyon""",[155]
"""Sciences Po / fonds CEVIPOF RÉ…","""saint-pierre""","[102, 1346]"
"""RÉPUBLIQUE FRANÇAISE - LIBERTÉ…","""mornet""",[84]
…,…,…
"""Claude HOARAU Égalité Responsa…","""hoarau""","[7, 846, 3701]"
"""Sciences Po / fonds CEVIPOF An…","""pihouee""",[2005]
"""Élie HOARAU Égalité Responsabi…","""hoarau""","[5, 3694]"


In [13]:
(df.select(pl.selectors.by_name("transcript","titulaire-nom","titulaire-nom-positions"))
    .filter(pl.col("titulaire-nom-positions").list.len() == 0)) # 50 rows where titulaire-nom does not appear in transcript

transcript,titulaire-nom,titulaire-nom-positions
str,str,list[i64]
"""ELECTIONS LEGISLATIVES DU 5 JU…","""Chevalier""",[]
"""DÉPARTEMENT DES ALPES-MARITIME…","""Vassalo""",[]
"""DONNEZ NOUS UN DEPUTE OM EN FI…","""Péron""",[]
"""Clément BESOMBES Conseiller Mu…","""Descombes""",[]
"""GEORGES CHAVANES Jean-Michel B…","""Chavanez""",[]
…,…,…
"""Sciences Po / fonds CEVIPOF Ma…","""Giovanelli""",[]
"""+ INITIATIVES COURAGE = RESULT…","""Guysel""",[]
"""Sciences Po / fonds CEVIPOF RE…","""Adevah-Poeuf""",[]


In [14]:
df = df.filter(pl.col("titulaire-nom-positions").list.len() > 0) # 3410 rows remaining

In [15]:
df.get_column("titulaire-nom-positions").list.len().describe()
# titulaire-nom appears 2 to 3 times in each transcript. So the number of rows and the number of appearances will have similar statistical properties

statistic,value
str,f64
"""count""",3410.0
"""null_count""",0.0
"""mean""",2.642229
"""std""",1.743405
"""min""",1.0
"""25%""",2.0
"""50%""",2.0
"""75%""",3.0
"""max""",25.0


# Gliner2

## NER : reconnaissance du nom

In [16]:
from gliner2 import GLiNER2
extractor = GLiNER2.from_pretrained("fastino/gliner2-base-v1")


You are using a model of type extractor to instantiate a model of type . This is not supported for all configurations of models and can yield errors.


🧠 Model Configuration
Encoder model      : microsoft/deberta-v3-base
Counting layer     : count_lstm_v2
Token pooling      : first


In [17]:
import spacy
# from spacy.util import filter_spans
nlp = spacy.blank("fr")  # tokenizer only

In [18]:
schema = extractor.create_schema().entities({
    "PER": "person",
    "ORG": "organization",
    "LOC": "location"
})


In [19]:
extractor.extract(df.item(0, "transcript"), schema, include_spans=True)

{'entities': {'PER': [{'text': 'François MITTERRAND',
    'start': 523,
    'end': 542},
   {'text': 'Bernard AULAGNE', 'start': 1232, 'end': 1247}],
  'ORG': [{'text': 'Front National', 'start': 1607, 'end': 1621},
   {'text': 'RPR', 'start': 2273, 'end': 2276},
   {'text': 'UDF', 'start': 614, 'end': 617},
   {'text': 'PC', 'start': 2287, 'end': 2289},
   {'text': 'Sauvegarde de la Bresse', 'start': 1295, 'end': 1318},
   {'text': 'Assemblée nationale', 'start': 1484, 'end': 1503},
   {'text': 'PS', 'start': 2283, 'end': 2285},
   {'text': 'for', 'start': 1944, 'end': 1947},
   {'text': 'CEVIPOF', 'start': 103, 'end': 110},
   {'text': 'socia', 'start': 1772, 'end': 1777}],
  'LOC': [{'text': 'France', 'start': 2774, 'end': 2780},
   {'text': 'Ain', 'start': 1389, 'end': 1392}]}}

## Recall pour chaque occurence

In [20]:
entities = extractor.extract(df.item(0, "transcript"), schema, include_spans=True)

In [21]:
entities["entities"]['PER']

[{'text': 'François MITTERRAND', 'start': 523, 'end': 542},
 {'text': 'Bernard AULAGNE', 'start': 1232, 'end': 1247}]

In [22]:
df.item(0,"titulaire-nom-positions")

""
i64
1240
2866


In [23]:
def find_position_in_entities(row):
    ents = extractor.extract(row["transcript"], schema, include_spans=True)
    i = row["position"]
    return {'PER' : any([(e['start'] <= i) & (e['end'] > i) for e in ents["entities"]['PER']]),
            'ORG' : any([(e['start'] <= i) & (e['end'] > i) for e in ents["entities"]['ORG']]),
            'LOC' : any([(e['start'] <= i) & (e['end'] > i) for e in ents["entities"]['LOC']])}

(df.explode(pl.col("titulaire-nom-positions")).head(2)
 .with_columns(
     pl.struct(["transcript","titulaire-nom-positions"])
         .struct.rename_fields(["transcript","position"])
         .map_elements(find_position_in_entities,
                       return_dtype=pl.Struct([pl.Field("PER",pl.Boolean),pl.Field("ORG",pl.Boolean),pl.Field("LOC",pl.Boolean)]))
         .alias("titulaire-nom-ents"))
 .select(pl.selectors.by_name("transcript","titulaire-nom","titulaire-nom-positions","titulaire-nom-ents"))
).unnest("titulaire-nom-ents")


transcript,titulaire-nom,titulaire-nom-positions,PER,ORG,LOC
str,str,i64,bool,bool,bool
"""Élections Législatives du 5 Ju…","""Aulagne""",1240,true,false,false
"""Élections Législatives du 5 Ju…","""Aulagne""",2866,false,false,false


In [24]:
# this is too slow : 4-5s per row

## Gliner Batch operation

In [25]:
extractor.batch_extract(
    df.head(3).get_column("transcript").to_list(), schema, include_spans=True, batch_size=8)

[{'entities': {'PER': [{'text': 'François MITTERRAND',
     'start': 523,
     'end': 542},
    {'text': 'Bernard AULAGNE', 'start': 1232, 'end': 1247}],
   'ORG': [{'text': 'Front National', 'start': 1607, 'end': 1621},
    {'text': 'RPR', 'start': 2273, 'end': 2276},
    {'text': 'UDF', 'start': 614, 'end': 617},
    {'text': 'PC', 'start': 2287, 'end': 2289},
    {'text': 'Sauvegarde de la Bresse', 'start': 1295, 'end': 1318},
    {'text': 'Assemblée nationale', 'start': 1484, 'end': 1503},
    {'text': 'PS', 'start': 2283, 'end': 2285},
    {'text': 'for', 'start': 1944, 'end': 1947},
    {'text': 'CEVIPOF', 'start': 103, 'end': 110},
    {'text': 'socia', 'start': 1772, 'end': 1777}],
   'LOC': [{'text': 'France', 'start': 2774, 'end': 2780},
    {'text': 'Ain', 'start': 1389, 'end': 1392}]}},
 {'entities': {'PER': [{'text': 'Jean-François MORTEL',
     'start': 744,
     'end': 764},
    {'text': 'PUJOL', 'start': 208, 'end': 213},
    {'text': 'JUQUIN', 'start': 193, 'end': 199}

In [26]:
# https://github.com/urchade/GLiNER/discussions/73 : batch processing does not appear to be faster

In [27]:
pbar = tqdm(total=df.height,ncols=100, mininterval=1.0, miniters=10,smoothing=0.1)
def find_position_in_entities(i,ents):
    return {'PER' : any([(e['start'] <= i) & (e['end'] > i) for e in ents["entities"]['PER']]),
            'ORG' : any([(e['start'] <= i) & (e['end'] > i) for e in ents["entities"]['ORG']]),
            'LOC' : any([(e['start'] <= i) & (e['end'] > i) for e in ents["entities"]['LOC']])}

def aux(row):
    pbar.update(1)
    ents = extractor.extract(row["transcript"], schema, include_spans=True)
    return [find_position_in_entities(i,ents) for i in row["positions"]]

#df.with_columns(
#    res = pl.struct(["transcript","titulaire-nom-positions"])
#        .struct.rename_fields(["transcript","positions"])
#        .map_elements(aux, pl.List(pl.Struct({'PER': pl.Boolean, 'ORG': pl.Boolean, 'LOC': pl.Boolean})))
#).explode(["titulaire-nom-positions","res"]).unnest("res").write_parquet("df_gliner.parquet")
# NOT RUN : takes approximately 2 hours

  0%|                                                                      | 0/3410 [00:19<?, ?it/s]

In [30]:
pl.read_parquet("df_gliner.parquet").select(pl.col("titulaire-nom"),pl.col("titulaire-nom-positions"),pl.col("PER"),pl.col("ORG"))

titulaire-nom,titulaire-nom-positions,PER,ORG
str,i64,bool,bool
"""Aulagne""",1240,true,false
"""Aulagne""",2866,false,false
"""Pujol""",208,true,false
"""Pujol""",3567,false,false
"""Boyon""",155,true,false
…,…,…,…
"""Virapoullé""",1360,true,false
"""Virapoullé""",1425,false,false
"""Vergès""",33,false,false


## Entités avec affiliation politique gauche / droite

schema = extractor.create_schema().entities({
    "PER": "person",
    "PER_LEFT": "person affiliated with left wing political parties",
    "PER_RIGHT": "person affiliated with right wing political parties", 
    "PER_GAUCHE": "personne affiliée avec un parti politique de gauche",
    "PER_DROITE": "personne affiliée avec un parti politique de droite",
})

pbar = tqdm(total=df.height,ncols=100, mininterval=1.0, miniters=10,smoothing=0.1)
def find_position_in_entities(i,ents):
    return {'PER' : any([(e['start'] <= i) & (e['end'] > i) for e in ents["entities"]['PER']]),
            'ORG' : any([(e['start'] <= i) & (e['end'] > i) for e in ents["entities"]['ORG']]),
            'LOC' : any([(e['start'] <= i) & (e['end'] > i) for e in ents["entities"]['LOC']])}

def aux(row):
    pbar.update(1)
    ents = extractor.extract(row["transcript"], schema, include_spans=True)
    return [find_position_in_entities(i,ents) for i in row["positions"]]

df.with_columns(
    res = pl.struct(["transcript","titulaire-nom-positions"])
        .struct.rename_fields(["transcript","positions"])
        .map_elements(aux, pl.List(pl.Struct({'PER': pl.Boolean, 'ORG': pl.Boolean, 'LOC': pl.Boolean})))
).explode(["titulaire-nom-positions","res"]).unnest("res").write_parquet("df_gliner.parquet")

In [28]:
schema = (extractor.create_schema()
    .entities({
        "PER": "person",
        "PER_LEFT": "person affiliated with left wing political parties",
        "PER_RIGHT": "person affiliated with right wing political parties", 
        "PER_GAUCHE": "personne affiliée avec un parti politique de gauche",
        "PER_DROITE": "personne affiliée avec un parti politique de droite",
        "ORG": "organization",
        "LOC": "location"
    })
    .classification("political ideology", ["left","right"])
    .structure("person")
        .field("name", dtype="str", description="name of the person")
        .field("political affiliation", dtype="str", choices=["left","right"])
)

extractor.extract(df.item(0, "transcript"), schema, include_spans=True)

{'person': [{'name': {'text': 'François MITTERRAND', 'start': 523, 'end': 542},
   'political affiliation': 'right'},
  {'name': None, 'political affiliation': 'right'}],
 'entities': {'PER': [{'text': 'Bernard AULAGNE', 'start': 1232, 'end': 1247},
   {'text': 'François MITTERRAND', 'start': 523, 'end': 542}],
  'PER_LEFT': [],
  'PER_RIGHT': [],
  'PER_GAUCHE': [],
  'PER_DROITE': [],
  'ORG': [{'text': 'RPR', 'start': 2273, 'end': 2276},
   {'text': 'UDF', 'start': 614, 'end': 617},
   {'text': 'Front National', 'start': 797, 'end': 811},
   {'text': 'PC', 'start': 2287, 'end': 2289},
   {'text': 'Sauvegarde de la Bresse', 'start': 1295, 'end': 1318},
   {'text': 'Assemblée nationale', 'start': 1484, 'end': 1503},
   {'text': 'PS', 'start': 2283, 'end': 2285}],
  'LOC': [{'text': 'Ain', 'start': 1389, 'end': 1392},
   {'text': 'France', 'start': 2774, 'end': 2780}]},
 'political ideology': 'right'}

In [33]:
e0 = extractor.extract(df.item(1, "transcript"), schema, include_spans=True)

In [36]:
e0.keys()

dict_keys(['person', 'entities', 'political ideology'])

In [55]:
pbar = tqdm(total=df.height,ncols=100, mininterval=1.0, miniters=10,smoothing=0.1)
def aux(row):
    pbar.update(1)
    return extractor.extract(row["transcript"], schema, include_spans=True)

df.with_columns(
    res = pl.struct(["transcript","titulaire-nom-positions"])
        .struct.rename_fields(["transcript","positions"])
        .map_elements(aux,
                      pl.Struct({
                          'person': pl.List(pl.Struct({
                              'name': pl.Struct({'text': pl.String,'start': pl.Int64,'end': pl.Int64}),
                              'political affiliation': pl.String
                          })),
                          'entities': pl.Struct({
                              'PER': pl.List(pl.Struct({'text': pl.String,'start': pl.Int64,'end': pl.Int64})),
                              'PER_LEFT': pl.List(pl.Struct({'text': pl.String,'start': pl.Int64,'end': pl.Int64})),
                              'PER_RIGHT': pl.List(pl.Struct({'text': pl.String,'start': pl.Int64,'end': pl.Int64})),
                              'PER_GAUCHE': pl.List(pl.Struct({'text': pl.String,'start': pl.Int64,'end': pl.Int64})),
                              'PER_DROITE': pl.List(pl.Struct({'text': pl.String,'start': pl.Int64,'end': pl.Int64})),
                              'ORG': pl.List(pl.Struct({'text': pl.String,'start': pl.Int64,'end': pl.Int64})),
                              'LOC': pl.List(pl.Struct({'text': pl.String,'start': pl.Int64,'end': pl.Int64})),
                          }),
                          'political ideology': pl.String
                              })
                     )
).write_parquet("df_gliner_full.parquet")

100%|█████████████████████████████████████████████████████████| 3410/3410 [1:41:31<00:00,  2.24s/it]

In [56]:
pl.read_parquet("df_gliner_full.parquet").unnest("res").select(pl.selectors.by_name("titulaire-nom","titulaire-nom-positions","person","entities","political ideology"))


titulaire-nom,titulaire-nom-positions,person,entities,political ideology
str,list[i64],list[struct[2]],struct[7],str
"""Aulagne""","[1240, 2866]","[{{""François MITTERRAND"",523,542},""right""}, {null,""right""}]","{[{""Bernard AULAGNE"",1232,1247}, {""François MITTERRAND"",523,542}],[],[],[],[],[{""RPR"",2273,2276}, {""UDF"",614,617}, … {""PS"",2283,2285}],[{""Ain"",1389,1392}, {""France"",2774,2780}]}","""right"""
"""Pujol""","[208, 3567]","[{{""JUQUIN"",193,199},""left""}, {{""PUJOL"",208,213},""right""}]","{[{""JUQUIN"",1472,1478}, {""PUJOL"",208,213}],[{""GAUCHE"",3479,3485}, {""JUQUIN"",193,199}],[{""PUJOL"",208,213}, {""Jean-François MORTEL"",744,764}],[],[],[{""CEVIPOF"",1707,1714}, {""CEVIPOFSciences"",1680,1695}, … {""Sciences Po"",1660,1671}],[{""Afrique"",496,503}]}","""left"""
"""Boyon""",[155],"[{{""Jacques BOYON"",147,160},""left""}, {{""Paul MORIN"",320,330},""left""}, {{""Jacques CHIRAC"",1745,1759},""right""}]","{[{""Paul MORIN"",4294,4304}, {""Jacques BOYON"",147,160}, {""Jacques CHIRAC"",1745,1759}],[],[{""Paul MORIN"",320,330}],[],[],[{""Assemblée Nationale"",807,826}, {""CEVIPOF"",20,27}, … {""Légion d'honneur"",273,289}],[{""BRESSE"",3598,3604}, {""BOURG"",3538,3543}, {""REVERMONT"",2998,3007}]}","""left"""
"""Saint-Pierre""","[102, 1346]","[{{""PIERRE FROMONT"",115,129},""left""}]","{[{""Pierre FROMONT"",1429,1443}, {""SAINT-PIERRE"",1346,1358}],[],[],[],[],[{""CEVIPOF"",20,27}],[{""Bourg"",823,828}, {""BRESSE"",1314,1320}, … {""AIN"",66,69}]}","""left"""
"""Mornet""",[84],"[{{""Lionel MORNET"",77,90},""left""}]","{[{""Lionel MORNET"",77,90}, {""Noëlle FAVIER"",162,175}],[],[{""F. Mitterrand"",987,1000}, {""Noëlle FAVIER"",162,175}],[{""Noëlle FAVIER"",162,175}],[{""F. Mitterrand"",987,1000}],[{""Assemblée Nationale"",1336,1355}, {""Parti Communiste Français"",3394,3419}, … {""Centre"",4009,4015}],[{""BOURG-EN-BRESSE"",146,161}, {""France"",457,463}]}","""left"""
…,…,…,…,…
"""Hoarau""","[7, 846, 3701]","[{{""Claude HOARAU"",0,13},""right""}, {{""André Thien Ah Koon"",561,580},""right""}]","{[{""Claude HOARAU"",3694,3707}, {""Philippe BERNE"",3720,3734}],[],[{""André Thien Ah Koon"",561,580}],[{""André Thien Ah Koon"",561,580}],[{""Claude HOARAU"",3694,3707}],[{""CEVIPOF"",2167,2174}, {""République"",1360,1370}, … {""CEVIPOFSciences"",2140,2155}],[{""Paris"",1911,1916}, {""France"",1501,1507}, {""Cilaos Saint-Louis"",202,220}]}","""right"""
"""Pihouee""",[2005],"[{{""Fred Lucien"",62,73},""right""}]","{[{""Paul VERGES"",1466,1477}],[],[],[],[],[{""Sciences Po / fonds CEVIPOF"",0,27}, {""parti communiste réunionnais"",1345,1373}],[{""St-Philippe"",164,175}, {""St-Joseph"",177,186}, … {""océan Indien"",1173,1185}]}","""left"""
"""Hoarau""","[5, 3694]","[{{""Élie HOARAU"",0,11},""right""}]","{[{""Élie HOARAU"",3689,3700}, {""Wilfrid Bertile"",636,651}],[],[{""Monsieur Pihouée"",733,749}],[],[],[{""CEVIPOF"",2154,2161}, {""République"",1345,1355}, {""CEVIPOFSciences"",2127,2142}],[{""Paris"",1757,1762}, {""France"",1485,1491}]}","""right"""


In [1]:
from tqdm.notebook import tqdm
import time

N = 500
pbar = tqdm(range(N), mininterval=1.0, miniters=10)
for i in pbar:
    pbar.set_description("Total: %s" % (i), refresh=False)
    # do something here
    pbar.update(1)
    time.sleep(0.01)


  0%|          | 0/500 [00:00<?, ?it/s]

In [88]:
def find_aux(row):
    return row["text"].find(row["pattern"])

pl.DataFrame({
    "text": ["apple pie", "banana split", "cherry tart"],
    "pattern": ["a", "split", "berry"]
}).with_columns(
    text_pattern = pl.struct(["text","pattern"])
    .map_elements(find_aux,return_dtype=pl.Int64)
)

text,pattern,text_pattern
str,str,i64
"""apple pie""","""a""",0
"""banana split""","""split""",7
"""cherry tart""","""berry""",-1


In [99]:
import re

def find_aux(row):
    return [match.start() for match in re.finditer(row["pattern"], row["text"])]

pl.DataFrame({
    "text": ["apple pie", "banana split", "cherry tart"],
    "pattern": ["a", "split", "berry"]
}).with_columns(
    text_pattern = pl.struct(["text","pattern"])
    .map_elements(find_aux,return_dtype=pl.List(pl.Int64))
)

text,pattern,text_pattern
str,str,list[i64]
"""apple pie""","""a""",[0]
"""banana split""","""split""",[7]
"""cherry tart""","""berry""",[]


In [114]:
import re

def find_aux(row):
    return [match.start() for match in re.finditer(row["pattern"], row["text"])]

pl.DataFrame({
    "toto": ["apple pie", "banana split", "cherry tart"],
    "tata": ["a", "split", "berry"]
}).with_columns(
    text_pattern = pl.struct(["toto","tata"])
        .struct.rename_fields(["text","pattern"])
        .map_elements(find_aux,return_dtype=pl.List(pl.Int64))
)

toto,tata,text_pattern
str,str,list[i64]
"""apple pie""","""a""",[0]
"""banana split""","""split""",[7]
"""cherry tart""","""berry""",[]
